# CART: Classification and Regression Trees — Student Performance

**Objective:** Understand and implement both branches of CART using a student-performance example.

- **Classification:** Predict whether a student will `Pass` or `Fail`.
- **Regression:** Predict a student's numerical exam score (`Marks`, from 0 to 100).

> **Dataset note:** This notebook uses an illustrative student dataset. The `Marks` column is generated from study hours, attendance, assignments, and a small random component for teaching regression. It is not real student-record data.

## What you will learn
1. Prepare and explore the dataset.
2. Understand binary splits and Gini impurity for classification.
3. Train and evaluate a `DecisionTreeClassifier`.
4. Train and evaluate a `DecisionTreeRegressor`.
5. Visualize both trees and make predictions for a new student.

Run the notebook from top to bottom.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, mean_absolute_error,
    mean_squared_error, r2_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Create the student performance dataset

The classification target is `Result`. We also create a numerical `Marks` target for the regression task. The generated marks are clipped to the range 0–100.

In [ ]:
data = {
    "Study_Hours": [1,2,3,1,4,5,2,6,3,7,1,4,5,2,6,3,7,8,2,5,4,9,1,6,8,3,7,4,9,2,5,8,6,1,7,3,9,4,8,2],
    "Attendance": [55,60,65,50,72,80,58,88,68,92,52,75,82,62,90,66,95,96,57,84,78,98,54,89,97,64,93,76,99,59,85,94,87,53,91,67,98,79,96,61],
    "Assignments": ["Low","Low","Medium","Low","Medium","High","Low","High","Medium","High","Low","Medium","High","Low","High","Medium","High","High","Low","High","Medium","High","Low","High","High","Medium","High","Medium","High","Low","High","High","High","Low","High","Medium","High","Medium","High","Low"],
    "Internet_Access": ["No","Yes","Yes","No","Yes","Yes","No","Yes","Yes","Yes","No","Yes","Yes","Yes","Yes","No","Yes","Yes","No","Yes","Yes","Yes","No","Yes","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","No","Yes","Yes","Yes","Yes","Yes","No"],
    "Result": ["Fail","Fail","Fail","Fail","Pass","Pass","Fail","Pass","Fail","Pass","Fail","Pass","Pass","Fail","Pass","Fail","Pass","Pass","Fail","Pass","Pass","Pass","Fail","Pass","Pass","Fail","Pass","Pass","Pass","Fail","Pass","Pass","Pass","Fail","Pass","Fail","Pass","Pass","Pass","Fail"]
}

df = pd.DataFrame(data)

# Create illustrative numerical marks for the regression exercise.
rng = np.random.default_rng(42)
assignment_points = df["Assignments"].map({"Low": 5, "Medium": 12, "High": 18})
internet_points = df["Internet_Access"].map({"No": 0, "Yes": 3})
noise = rng.normal(loc=0, scale=5, size=len(df))

df["Marks"] = (
    15
    + df["Study_Hours"] * 5
    + df["Attendance"] * 0.25
    + assignment_points
    + internet_points
    + noise
).clip(0, 100).round(1)

display(df.head(10))
print("Dataset shape:", df.shape)

## 3. Explore the data

In [ ]:
print("Data types and non-null counts:")
df.info()

print("\\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))

print("\\nClassification target distribution:")
display(df["Result"].value_counts())

print("\\nNumerical summary:")
display(df[["Study_Hours", "Attendance", "Marks"]].describe())

# Part A — CART Classification

## 4. Select features and classification target

We predict `Result` (`Pass` / `Fail`). We deliberately exclude `Marks` from the features so the model does not use a target-related score to predict the result.

In [ ]:
feature_columns = ["Study_Hours", "Attendance", "Assignments", "Internet_Access"]
X_class = df[feature_columns]
y_class = df["Result"]

print("Features:")
display(X_class.head())
print("Target:")
display(y_class.head())

## 5. Understand Gini impurity

Gini impurity measures how mixed the classes are in a node.

- Gini = 0 means the node contains only one class.
- A larger value means the node contains a more mixed set of classes.
- CART evaluates binary splits and prefers splits that reduce weighted impurity.

In [ ]:
def gini_impurity(labels):
    proportions = labels.value_counts(normalize=True)
    return 1 - np.sum(proportions ** 2)

parent_gini = gini_impurity(y_class)
print("Parent Gini impurity:", round(parent_gini, 4))

## 6. Calculate weighted child Gini for an example binary split

In [ ]:
# Example split: Study_Hours <= 4 versus Study_Hours > 4
left_mask = X_class["Study_Hours"] <= 4
left_labels = y_class[left_mask]
right_labels = y_class[~left_mask]

weighted_child_gini = (
    len(left_labels) / len(y_class) * gini_impurity(left_labels)
    + len(right_labels) / len(y_class) * gini_impurity(right_labels)
)

gini_reduction = parent_gini - weighted_child_gini

print("Left node Gini:", round(gini_impurity(left_labels), 4))
print("Right node Gini:", round(gini_impurity(right_labels), 4))
print("Weighted child Gini:", round(weighted_child_gini, 4))
print("Gini reduction:", round(gini_reduction, 4))

## 7. Split the classification data

We use an 80:20 train-test split. `stratify=y_class` keeps the Pass/Fail proportions similar in both sets.

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class, y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print("Training rows:", len(Xc_train))
print("Testing rows:", len(Xc_test))
print("\\nTraining class proportions:")
display(yc_train.value_counts(normalize=True).round(2))
print("Testing class proportions:")
display(yc_test.value_counts(normalize=True).round(2))

## 8. Preprocess features and build the CART classifier

`OneHotEncoder` converts categorical columns into numeric columns. The preprocessing is inside a `Pipeline`, so it is fitted on the training data only.

For classification, `criterion="gini"` uses Gini impurity to choose splits.

In [ ]:
numeric_features = ["Study_Hours", "Attendance"]
categorical_features = ["Assignments", "Internet_Access"]

classification_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)

cart_classifier = Pipeline(steps=[
    ("preprocessor", classification_preprocessor),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=4,
        random_state=42
    ))
])

cart_classifier.fit(Xc_train, yc_train)
print("CART classifier training completed.")

## 9. Classification predictions and evaluation

In [ ]:
yc_pred = cart_classifier.predict(Xc_test)

classification_results = pd.DataFrame({
    "Actual Result": yc_test.values,
    "Predicted Result": yc_pred
})
display(classification_results)

accuracy = accuracy_score(yc_test, yc_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\\nClassification report:")
print(classification_report(yc_test, yc_pred, zero_division=0))

## 10. Classification confusion matrix

In [ ]:
cm = confusion_matrix(yc_test, yc_pred, labels=["Fail", "Pass"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail", "Pass"])
disp.plot(cmap="Blues")
plt.title("CART Classifier — Confusion Matrix")
plt.show()

## 11. Visualize the classification tree

In [ ]:
fitted_class_preprocessor = cart_classifier.named_steps["preprocessor"]
class_feature_names = fitted_class_preprocessor.get_feature_names_out()
class_tree = cart_classifier.named_steps["model"]

plt.figure(figsize=(20, 10))
plot_tree(
    class_tree,
    feature_names=class_feature_names,
    class_names=[str(c) for c in class_tree.classes_],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("CART Classification Tree — Student Pass/Fail")
plt.show()

## 12. Predict for a new student — Classification

In [ ]:
new_student = pd.DataFrame({
    "Study_Hours": [6],
    "Attendance": [85],
    "Assignments": ["High"],
    "Internet_Access": ["Yes"]
})

class_prediction = cart_classifier.predict(new_student)
print("Predicted student result:", class_prediction[0])

# Part B — CART Regression

## 13. Select features and regression target

For regression, the target is `Marks`, a continuous numerical value. We use the same input features, but this time the model predicts a number rather than a category.

In [ ]:
X_reg = df[feature_columns]
y_reg = df["Marks"]

display(pd.concat([X_reg.head(), y_reg.head().rename("Marks")], axis=1))
print("Target type:", y_reg.dtype)

## 14. Split the regression data

We use an 80:20 split. We do not use stratification because `Marks` is a continuous numerical target.

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(Xr_train))
print("Testing rows:", len(Xr_test))

## 15. Build and train the CART regressor

For regression, CART chooses splits that reduce prediction error. Scikit-learn's `criterion="squared_error"` uses mean squared error to evaluate splits.

In [ ]:
regression_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)

cart_regressor = Pipeline(steps=[
    ("preprocessor", regression_preprocessor),
    ("model", DecisionTreeRegressor(
        criterion="squared_error",
        max_depth=4,
        random_state=42
    ))
])

cart_regressor.fit(Xr_train, yr_train)
print("CART regressor training completed.")

## 16. Regression predictions and evaluation

Regression metrics:
- **MAE:** Average absolute prediction error; lower is better.
- **MSE:** Average squared prediction error; lower is better.
- **RMSE:** Square root of MSE, in the same units as marks; lower is better.
- **R²:** Proportion of target variance explained by the model; closer to 1 is generally better.

In [ ]:
yr_pred = cart_regressor.predict(Xr_test)

regression_results = pd.DataFrame({
    "Actual Marks": yr_test.values,
    "Predicted Marks": np.round(yr_pred, 1)
})
regression_results["Absolute Error"] = (
    regression_results["Actual Marks"] - regression_results["Predicted Marks"]
).abs()
display(regression_results)

mae = mean_absolute_error(yr_test, yr_pred)
mse = mean_squared_error(yr_test, yr_pred)
rmse = np.sqrt(mse)
r2 = r2_score(yr_test, yr_pred)

print(f"MAE : {mae:.2f}")
print(f"MSE : {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.3f}")

## 17. Actual vs predicted marks

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(yr_test, yr_pred)
minimum = min(yr_test.min(), yr_pred.min())
maximum = max(yr_test.max(), yr_pred.max())
plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")
plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.title("CART Regression — Actual vs Predicted Marks")
plt.grid(True, alpha=0.3)
plt.show()

## 18. Visualize the regression tree

In [ ]:
fitted_reg_preprocessor = cart_regressor.named_steps["preprocessor"]
reg_feature_names = fitted_reg_preprocessor.get_feature_names_out()
reg_tree = cart_regressor.named_steps["model"]

plt.figure(figsize=(20, 10))
plot_tree(
    reg_tree,
    feature_names=reg_feature_names,
    filled=True,
    rounded=True,
    impurity=True,
    fontsize=8
)
plt.title("CART Regression Tree — Student Marks")
plt.show()

## 19. Predict marks for a new student — Regression

In [ ]:
new_student = pd.DataFrame({
    "Study_Hours": [6],
    "Attendance": [85],
    "Assignments": ["High"],
    "Internet_Access": ["Yes"]
})

marks_prediction = cart_regressor.predict(new_student)
print(f"Predicted student marks: {marks_prediction[0]:.1f} / 100")

# Part C — Final Comparison and Conclusion

## 20. CART Classifier vs CART Regressor

| Aspect | CART Classification | CART Regression |
|---|---|---|
| Scikit-learn class | `DecisionTreeClassifier` | `DecisionTreeRegressor` |
| Target | Categorical | Continuous numerical |
| This notebook predicts | Pass / Fail | Marks |
| Split criterion used | Gini impurity | Squared error (MSE) |
| Evaluation | Accuracy, precision, recall, F1 | MAE, MSE, RMSE, R² |
| Output | Class label | Numerical value |

## 21. Key takeaways

- CART stands for **Classification and Regression Trees**.
- CART builds decision trees using binary splits.
- A classifier predicts a category or class.
- A regressor predicts a continuous numerical value.
- `DecisionTreeClassifier(criterion="gini")` implements the classification example.
- `DecisionTreeRegressor(criterion="squared_error")` implements the regression example.
- The preprocessing and model are kept in pipelines to avoid fitting preprocessing steps on test data.

**Conclusion:** This notebook demonstrates both classification and regression using CART on an illustrative student-performance dataset. The classification task predicts Pass/Fail, while the regression task estimates numerical marks.